In [8]:
from datasets import load_dataset
def load_wildguard():
    wildguard_test = load_dataset("allenai/wildguardmix", "wildguardtest")["test"]
    wildguard_test = wildguard_test.to_pandas()
    wildguard_test = wildguard_test.dropna()

    prompts = wildguard_test["prompt"].values.tolist()
    labels = wildguard_test["prompt_harm_label"].values.tolist()
    map2id = {"unharmful": 0, "harmful": 1}
    labels= [map2id[x] for x in labels]
    return prompts,labels,wildguard_test

In [9]:
real_prompt, real_labels,wild_df = load_wildguard()

Using the latest cached version of the dataset since allenai/wildguardmix couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'wildguardtest' at /home/unica/.cache/huggingface/datasets/allenai___wildguardmix/wildguardtest/0.0.0/d29c47f41c8b51348b5c8e8c81c039b3132b66d1 (last modified on Mon May 26 10:32:15 2025).


In [10]:
import pandas as pd
import os
from glob import glob
from sklearn.metrics import accuracy_score, f1_score

def accuracy_f1(real, preds):
    return {
        "ACC": round(accuracy_score(real, preds), 4),
        "F1": round(f1_score(real, preds), 4)
    }

def wildguard_scores(pred_df):
    wild_adv=wild_df[wild_df["adversarial"]==True]    
    pred_adv=pred_df[pred_df["text"].isin(wild_adv["prompt"])]    
    wild_van=wild_df[wild_df["adversarial"]==False]
    pred_van=pred_df[pred_df["text"].isin(wild_van["prompt"])]
    
    overall=accuracy_f1(pred_df["real"].values.tolist(),pred_df["pred"].values.tolist())
    vanilla=accuracy_f1(pred_van["real"].values.tolist(),pred_van["pred"].values.tolist())
    adversarial=accuracy_f1(pred_adv["real"].values.tolist(),pred_adv["pred"].values.tolist())
    # Restituiamo una lista di dict per facilitare il DataFrame
    return [
        {"Method": "Overall", **overall},
        {"Method": "Vanilla", **vanilla},
        {"Method": "Adversarial", **adversarial}
    ]

def compute_scores(folder):
    # Path base (usiamo fold-0 solo per listare i modelli disponibili)
    base_path = f"../output/fold-0/parsed/{folder}/"
    models = [x for x in os.listdir(base_path) if not x.startswith(".")]

    for model in models:
        all_folds_results = []
        all_folds_wild = []

        for fold in range(3):
            preds_files = glob(f"../output/fold-{fold}/parsed/{folder}/{model}/*.json")
            
            for f_path in preds_files:
                dataset_name = os.path.basename(f_path).replace(".json", "")
                pred_df = pd.read_json(f_path)
                
                if dataset_name != "WildGuard":
                    # IN-DOMAIN / STANDARD OOD
                    metrics = accuracy_f1(pred_df["real"], pred_df["pred"])
                    metrics.update({"dataset": dataset_name, "fold": fold})
                    all_folds_results.append(metrics)
                else:
                    # WILDGUARD (OUT-OF-DOMAIN con split interni)
                    metrics_list = wildguard_scores(pred_df)
                    for m in metrics_list:
                        m.update({"dataset": "WildGuard", "fold": fold})
                        all_folds_wild.append(m)

        print(f"\n--- REPORT FOR MODEL: {model} ---")
        
        # 1. Processing IN-DOMAIN (o altri dataset standard)
        if all_folds_results:
            df_in = pd.DataFrame(all_folds_results)
            # Raggruppiamo per dataset e calcoliamo media e std
            in_summary = df_in.groupby("dataset")[["ACC", "F1"]].agg(["mean", "std"]).round(3)
            print("\n[IN-DOMAIN / STANDARD DATASETS]")
            print(in_summary)

        # 2. Processing WILDGUARD (OOD con split)
        if all_folds_wild:
            df_wild = pd.DataFrame(all_folds_wild)
            # Raggruppiamo per il tipo di split (Method)
            wild_summary = df_wild.groupby("Method")[["ACC", "F1"]].agg(["mean", "std"]).round(3)
            print("\n[OUT-OF-DOMAIN: WILDGUARD]")
            print(wild_summary)
            
        print("-" * 50)



### FINE-TUNED

In [16]:
compute_scores("FT")


--- REPORT FOR MODEL: nuner ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC            F1       
            mean    std   mean    std
dataset                              
Aegis      0.807  0.013  0.838  0.012
OrBench    0.675  0.020  0.000  0.000
Remedy     0.941  0.014  0.939  0.015
ToxicChat  0.953  0.003  0.701  0.021

[OUT-OF-DOMAIN: WILDGUARD]
               ACC            F1       
              mean    std   mean    std
Method                                 
Adversarial  0.696  0.008  0.486  0.029
Overall      0.779  0.004  0.690  0.008
Vanilla      0.851  0.010  0.818  0.012
--------------------------------------------------

--- REPORT FOR MODEL: gemma2 ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC            F1       
            mean    std   mean    std
dataset                              
Aegis      0.848  0.004  0.875  0.003
OrBench    0.749  0.060  0.000  0.000
Remedy     0.976  0.002  0.976  0.002
ToxicChat  0.954  0.004  0.742  0.015

[OUT-OF-DOMAIN:

### ZERO-SHOT


In [17]:
compute_scores("ZERO")


--- REPORT FOR MODEL: gemma2 ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.813 NaN  0.855 NaN
OrBench    0.147 NaN  0.000 NaN
Remedy     0.899 NaN  0.902 NaN
ToxicChat  0.924 NaN  0.618 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           
Adversarial  0.775 NaN  0.769 NaN
Overall      0.850 NaN  0.842 NaN
Vanilla      0.916 NaN  0.908 NaN
--------------------------------------------------

--- REPORT FOR MODEL: llama3.3-70 ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.816 NaN  0.844 NaN
OrBench    0.500 NaN  0.000 NaN
Remedy     0.920 NaN  0.917 NaN
ToxicChat  0.958 NaN  0.701 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           


### GUARDIANS

In [18]:
compute_scores("GUARD")


--- REPORT FOR MODEL: shieldgemma ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.741 NaN  0.757 NaN
OrBench    0.737 NaN  0.000 NaN
Remedy     0.821 NaN  0.790 NaN
ToxicChat  0.959 NaN  0.668 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           
Adversarial  0.678 NaN  0.406 NaN
Overall      0.720 NaN  0.559 NaN
Vanilla      0.758 NaN  0.660 NaN
--------------------------------------------------

--- REPORT FOR MODEL: duo_1.5 ---

[IN-DOMAIN / STANDARD DATASETS]
             ACC         F1    
            mean std   mean std
dataset                        
Aegis      0.780 NaN  0.803 NaN
OrBench    0.776 NaN  0.000 NaN
Remedy     0.882 NaN  0.871 NaN
ToxicChat  0.955 NaN  0.642 NaN

[OUT-OF-DOMAIN: WILDGUARD]
               ACC         F1    
              mean std   mean std
Method                           

### MULTITURN (COSAFE)

In [19]:
import json
def evaluate_multiturn(mode):
    base = "../output/multiTurn" if mode == "FT" else "../output/multiTurn/ZERO"
    if not os.path.exists(base):
        print(f"No results found for {mode}")
        return
    models = sorted([x for x in os.listdir(base)
                     if os.path.isdir(os.path.join(base, x)) and not x.startswith(".")])
    for model in models:
        model_path = os.path.join(base, model)
        rows = []
        for f in sorted(glob(f"{model_path}/*.json")):
            category = os.path.basename(f).replace(".json", "")
            data = json.load(open(f))
            preds = [item["pred"] for item in data]
            accuracy= round(sum(preds) / len(preds), 4) if preds else 0.0
            rows.append({"Category": category, "Accuracy": accuracy})
        if rows:
            df = pd.DataFrame(rows)
            overall = round(df["Accuracy"].mean(), 4)
            print(f"\n--- [{mode}] MODEL: {model} ---")
            print(df.to_string(index=False))
            print(f"  Overall: {overall}")

evaluate_multiturn("FT")
evaluate_multiturn("ZERO")



--- [FT] MODEL: gemma2 ---
                                       Category  Accuracy
                                   animal_abuse      0.80
                                    child_abuse      0.92
                  controversial_topics,politics      0.88
            discrimination,stereotype,injustice      0.93
            drug_abuse,weapons,banned_substance      0.94
           financial_crime,property_crime,theft      0.93
                 hate_speech,offensive_language      0.90
misinformation_regarding_ethics,laws_and_safety      0.77
                 non_violent_unethical_behavior      0.89
                              privacy_violation      0.92
                                      self_harm      0.91
                sexually_explicit,adult_content      0.92
                      terrorism,organized_crime      0.87
        violence,aiding_and_abetting,incitement      0.88
  Overall: 0.89

--- [FT] MODEL: llama3.1-8 ---
                                       Category  Accur